# In-situ recovery of simulated MI edge maps

Run through `run_benchmarks.py`. Full mode recomputes NMF-LR, COMMOT and ScCChain on the selected sample. Plot-only requires a verified score cache from that same workflow. See README for inputs and paper panels.


In [ ]:
from pathlib import Path
import shutil
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
from scipy.optimize import linear_sum_assignment
from scipy.stats import ranksums, spearmanr


import os
import simulation_benchmark_utils as sim_utils

METHOD_ORDER = sim_utils.METHOD_ORDER
SETTING_LIST = sim_utils.SETTING_LIST
get_local_result_dir = sim_utils.get_local_result_dir

# ------------------------------------------------------------
# 0. Select one simulation sample
# ------------------------------------------------------------
SETTING_INDEX = globals().get("SETTING_INDEX", 0)
EXPERIMENT_INDEX = globals().get("EXPERIMENT_INDEX", 18)
METHODS_TO_SHOW = METHOD_ORDER.copy()

RUN_NMF_LR = True
RUN_COMMOT = True
RUN_SCCCHAIN = True
LOAD_COMPATIBLE_SPACIA = False
SHOW_GROUND_TRUTH = True
CELL_SIZE = 2
TOPK_SCALE = 1.0
EDGE_LINEWIDTH = 0.8
SAVE_FIGURE = True

PLOT_ONLY = globals().get("PLOT_ONLY", False)
setting_name = SETTING_LIST[SETTING_INDEX]
SAMPLE_OUT_DIR = sim_utils.OUTPUT_ROOT / "SpiderNet" / setting_name / f"Experiment_{EXPERIMENT_INDEX}" / "SpiderNet_Result_Mode_cell_class"
SAMPLE_OUT_DIR.mkdir(parents=True, exist_ok=True)
payload = sim_utils.load_simulation_inputs(
    setting_idx=SETTING_INDEX,
    experiment_idx=EXPERIMENT_INDEX,
)
edge_index = np.asarray(payload["edge_index"], dtype=np.int64)
edgemeta_data = payload["edgemeta_data"].reset_index(drop=True)
spatial_location = np.asarray(payload["spatial_location"], dtype=float)
reference_pairs = pd.DataFrame(
    edge_index, columns=["sender_index", "receiver_index"]
)
truth_labels = edgemeta_data["MetaItype"].astype(str).to_numpy()


def _edge_score_table(mi1_scores, mi2_scores, source):
    score_table = reference_pairs.copy()
    score_table["MI-1"] = np.asarray(mi1_scores, dtype=float)
    score_table["MI-2"] = np.asarray(mi2_scores, dtype=float)
    score_table.attrs["source"] = source
    return score_table


def _match_latent_components_one_to_one(factor_matrix, method_name):
    """Match two latent components to MI-1/MI-2 as specified in Note A.2."""
    factor_matrix = np.asarray(factor_matrix, dtype=float)
    if factor_matrix.ndim != 2 or factor_matrix.shape[0] != len(edge_index):
        raise ValueError(
            f"{method_name} factor matrix has shape {factor_matrix.shape}; expected "
            f"({len(edge_index)}, n_components)."
        )
    if factor_matrix.shape[1] < 2:
        raise ValueError(f"{method_name} must provide at least two components.")

    correlations = np.array([
        [
            spearmanr(
                (truth_labels == mi_name).astype(float),
                factor_matrix[:, component_index],
            ).statistic
            for component_index in range(factor_matrix.shape[1])
        ]
        for mi_name in ["MI-1", "MI-2"]
    ])
    correlations = np.nan_to_num(correlations, nan=-np.inf)
    mi_indices, component_indices = linear_sum_assignment(
        correlations, maximize=True
    )
    component_by_mi = dict(zip(mi_indices, component_indices))
    print(
        f"[{method_name}] one-to-one Spearman component match: "
        f"MI-1 <- {component_by_mi[0]}, MI-2 <- {component_by_mi[1]}"
    )
    return _edge_score_table(
        factor_matrix[:, component_by_mi[0]],
        factor_matrix[:, component_by_mi[1]],
        source="recomputed on the selected reference graph",
    )


def _load_compatible_external_scores(method_name, analysis_root):
    """Load external scores only when their recorded edge graph is identical."""
    sample_name = f"{setting_name}_Experiment_{EXPERIMENT_INDEX}"
    sample_dir = Path(analysis_root) / setting_name / f"Experiment_{EXPERIMENT_INDEX}"
    edge_path = sample_dir / f"{sample_name}_reference_edge_index.csv"
    score_path = sample_dir / f"{sample_name}_{method_name}_edge_scores.npy"

    if not edge_path.is_file() or not score_path.is_file():
        print(f"[{method_name}] compatible external score files were not found; skipping.")
        return None

    saved_edges = pd.read_csv(edge_path)[
        ["sender_index", "receiver_index"]
    ].to_numpy(dtype=np.int64)
    if not np.array_equal(saved_edges, edge_index):
        overlap = len(set(map(tuple, saved_edges)) & set(map(tuple, edge_index)))
        print(
            f"[{method_name}] incompatible edge graph: {overlap}/{len(edge_index)} "
            "directed edges overlap with the selected sample; skipping. Rerun this "
            "method for the current simulation data before plotting it."
        )
        return None

    scores = np.load(score_path)
    if method_name == "ScCChain":
        return _match_latent_components_one_to_one(scores, method_name)
    if scores.ndim != 2 or scores.shape != (len(edge_index), 2):
        raise ValueError(
            f"{method_name} score matrix has shape {scores.shape}; expected "
            f"({len(edge_index)}, 2)."
        )
    return _edge_score_table(
        scores[:, 0], scores[:, 1], source="compatible external output"
    )


def _find_sccchain_runner():
    runner = Path(sim_utils.__file__).resolve().with_name("ScCChain_runner.jl")
    if not runner.is_file():
        raise FileNotFoundError(f"Missing local ScCChain runner: {runner}")
    return runner


def recompute_sccchain_on_current_sample():
    """Run ScCChain on the current h5ad, then align its radius graph to the 10-NN reference graph."""
    julia_executable = shutil.which("julia")
    if julia_executable is None:
        raise FileNotFoundError(
            "Julia is required for ScCChain. Install Julia and the ScCChain package."
        )

    sample_name = f"{setting_name}_Experiment_{EXPERIMENT_INDEX}"
    work_dir = (
        (sim_utils.OUTPUT_ROOT / "ScCChain_analysis") / setting_name
        / f"Experiment_{EXPERIMENT_INDEX}" / "insitu_recomputed"
    )
    work_dir.mkdir(parents=True, exist_ok=True)
    score_csv = work_dir / f"{sample_name}_ScCChain_edge_program_scores.csv"
    lr_db_csv = work_dir / f"{sample_name}_lr_db.csv"
    manifest_csv = work_dir / f"{sample_name}_manifest.csv"

    gene_meta = payload["genemeta_data"].reset_index()
    ligand_rows = gene_meta[gene_meta["Gene_type"].astype(str) == "Ligand"].copy()
    lr_db = pd.DataFrame({
        "ligand": ligand_rows["Gene_name"].astype(str),
        "receptor": ligand_rows["Associated_Ligand_or_Receptor"].astype(str),
        "pathway": ligand_rows["Associated_metaItype"].astype(str),
    })
    lr_db.to_csv(lr_db_csv, index=False)

    h5ad_path = Path(payload["data_dir"]) / "adata_simulation.h5ad"
    pd.DataFrame([{
        "sample_index": EXPERIMENT_INDEX,
        "sample_id": sample_name,
        "sample_name": sample_name,
        "sample_tag": "insitu",
        "h5ad_path": h5ad_path.as_posix(),
        "score_csv": score_csv.as_posix(),
    }]).to_csv(manifest_csv, index=False)

    command = [
        julia_executable, str(_find_sccchain_runner()),
        "--manifest", manifest_csv.as_posix(),
        "--lr-db-csv", lr_db_csv.as_posix(),
        "--result-root", work_dir.as_posix(),
        "--n-programs", "2", "--seed", "42",
        "--knn-k", "10", "--alpha", "0.00002",
        "--overwrite", "--fail-fast",
    ]
    completed = subprocess.run(
        command, check=True, text=True, capture_output=True
    )
    print(completed.stdout)

    raw_scores = pd.read_csv(score_csv)
    raw_pairs = raw_scores.iloc[:, :2].apply(pd.to_numeric).to_numpy(dtype=np.int64) - 1
    overlap = len(set(map(tuple, raw_pairs)) & set(map(tuple, edge_index)))
    print(
        f"[ScCChain] freshly recomputed radius graph overlaps {overlap}/"
        f"{len(edge_index)} reference edges; absent reference edges receive zero, "
        "matching the original benchmark conversion."
    )
    factors = sim_utils.load_sccchain_scores_to_reference_edges(
        score_csv, edge_index
    )
    return _match_latent_components_one_to_one(factors, "ScCChain")


# ------------------------------------------------------------
# 1. Build method scores on the selected sample's edge graph
# ------------------------------------------------------------
cache_path = SAMPLE_OUT_DIR / "insitu_scores.npz"
cache_parameters = dict(setting=SETTING_INDEX, experiment=EXPERIMENT_INDEX,
    run_nmf_lr=RUN_NMF_LR, run_commot=RUN_COMMOT, run_sccchain=RUN_SCCCHAIN,
    load_compatible_spacia=LOAD_COMPATIBLE_SPACIA, methods=METHODS_TO_SHOW)
if PLOT_ONLY:
    method_edge_scores = sim_utils.load_insitu_cache(cache_path, payload, cache_parameters)
else:
    method_edge_scores = {}

    spidernet_path = (
        get_local_result_dir("SpiderNet", setting_name, EXPERIMENT_INDEX)
        / "EdgeProgramScores.csv"
    )
    spidernet_raw = sim_utils.read_edge_scores_csv(spidernet_path)
    spidernet_scores = sim_utils.align_edge_scores_to_reference(
        spidernet_raw,
        edge_index=edge_index,
        method_name="SpiderNet",
        verbose=True,
    )
    if spidernet_scores.attrs.get("alignment_strategy") != "direct_pair_match":
        raise ValueError(
            "SpiderNet EdgeProgramScores.csv does not match the selected sample's edge graph."
        )
    spidernet_scores.attrs["source"] = "saved SpiderNet output on the reference graph"
    method_edge_scores["SpiderNet"] = spidernet_scores

    if RUN_NMF_LR and "NMF-LR" in METHODS_TO_SHOW:
        nmf_factors, _ = sim_utils.compute_nmf_lr_scores(
            payload["cellpair_lr"], n_components=2
        )
        method_edge_scores["NMF-LR"] = _match_latent_components_one_to_one(
            nmf_factors, "NMF-LR"
        )

    if RUN_COMMOT and "COMMOT" in METHODS_TO_SHOW:
        commot_scores = sim_utils.compute_commot_scores(
            expression=payload["expression"],
            gene_names=payload["gene_names"],
            cell_names=payload["cell_names"],
            spatial_pos=spatial_location,
            edge_index_np=edge_index,
        )
        method_edge_scores["COMMOT"] = _edge_score_table(
            commot_scores[:, 0],
            commot_scores[:, 1],
            source="recomputed on the selected reference graph",
        )

    if RUN_SCCCHAIN and "ScCChain" in METHODS_TO_SHOW:
        method_edge_scores["ScCChain"] = recompute_sccchain_on_current_sample()

    if LOAD_COMPATIBLE_SPACIA and "Spacia" in METHODS_TO_SHOW:
        spacia_scores = _load_compatible_external_scores(
            "Spacia", sim_utils.SPACIA_ANALYSIS_ROOT
        )
        if spacia_scores is not None:
            method_edge_scores["Spacia"] = spacia_scores

    sim_utils.save_insitu_cache(cache_path, payload, cache_parameters, method_edge_scores)

print("Resolved data root:", payload.get("data_root"))
print("Methods available for plotting:", [
    method_name for method_name in METHODS_TO_SHOW
    if method_name in method_edge_scores
])


In [ ]:
# ------------------------------------------------------------
# 2. Validate the common edge coordinate system
# ------------------------------------------------------------
for method_name, score_df in method_edge_scores.items():
    method_pairs = score_df[["sender_index", "receiver_index"]].reset_index(drop=True)
    if not method_pairs.equals(reference_pairs.reset_index(drop=True)):
        raise ValueError(f"{method_name} scores are not on the reference edge graph.")
    if len(score_df) != len(edgemeta_data):
        raise ValueError(
            f"{method_name} score/truth length mismatch: {len(score_df)} vs "
            f"{len(edgemeta_data)}."
        )


In [ ]:
# ------------------------------------------------------------
# 3. In-situ recovery plot helpers
# ------------------------------------------------------------
MI_NAMES = ["MI-1", "MI-2"]
MI_COLORS = {"MI-1": "#ED2087", "MI-2": "#FAA51A"}
MI_CMAPS = {
    mi_name: LinearSegmentedColormap.from_list(
        f"white_to_{mi_name}", ["white", color], N=256
    )
    for mi_name, color in MI_COLORS.items()
}
TP_FP_CMAP = LinearSegmentedColormap.from_list(
    "false_white_true", ["#3D3BF3", "white", "#FF2929"], N=256
)
TP_FP_NORM = Normalize(vmin=-1.0, vmax=1.0)


def build_truth_edge_table():
    truth_df = reference_pairs.copy()
    truth_df["truth_label"] = truth_labels
    return truth_df[truth_df["truth_label"].isin(MI_NAMES)].reset_index(drop=True)


def build_topk_prediction_table(edge_scores_df, truth_edge_df):
    """Select predictions from the full tissue edge pool; truth sets only k."""
    selected_frames = []
    for mi_name in MI_NAMES:
        scores = np.nan_to_num(
            edge_scores_df[mi_name].to_numpy(dtype=float),
            nan=-np.inf, posinf=np.inf, neginf=-np.inf,
        )
        k_truth = int((truth_edge_df["truth_label"] == mi_name).sum())
        k_use = max(1, min(len(scores), int(round(k_truth * TOPK_SCALE))))
        selected_indices = np.argsort(scores)[::-1][:k_use]
        selected = edge_scores_df.iloc[selected_indices][
            ["sender_index", "receiver_index"]
        ].copy()
        selected["plot_label"] = mi_name
        selected["score"] = scores[selected_indices]
        finite_scores = np.where(np.isfinite(selected["score"]), selected["score"], 0.0)
        score_min = float(np.min(finite_scores))
        score_range = float(np.max(finite_scores) - score_min)
        selected["score_norm"] = (
            (finite_scores - score_min) / score_range if score_range > 0
            else np.ones(len(selected), dtype=float)
        )
        selected_frames.append(selected)
    return pd.concat(selected_frames, ignore_index=True)


def _draw_arrows(ax, edge_df, colors, linewidths):
    """Draw directed edges with the original annotate-arrow geometry."""
    for edge, edge_color, linewidth in zip(
        edge_df.itertuples(index=False), colors, linewidths
    ):
        sender_index = int(edge.sender_index)
        receiver_index = int(edge.receiver_index)
        ax.annotate(
            "",
            xy=(spatial_location[receiver_index, 0], spatial_location[receiver_index, 1]),
            xytext=(spatial_location[sender_index, 0], spatial_location[sender_index, 1]),
            arrowprops=dict(
                arrowstyle="->,head_length=0.05,head_width=0.05",
                color=edge_color,
                lw=linewidth,
            ),
        )


def format_spatial_axis(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")


def plot_ground_truth(ax, truth_edge_df):
    ax.scatter(
        spatial_location[:, 0], spatial_location[:, 1],
        c="lightgrey", s=CELL_SIZE,
    )
    colors = [MI_COLORS[label] for label in truth_edge_df["truth_label"]]
    linewidths = np.full(len(truth_edge_df), EDGE_LINEWIDTH)
    _draw_arrows(ax, truth_edge_df, colors=colors, linewidths=linewidths)
    ax.set_title("Ground truth")
    format_spatial_axis(ax)


def plot_method(ax, method_name, edge_scores_df, truth_edge_df):
    predicted_edges = build_topk_prediction_table(edge_scores_df, truth_edge_df)
    strengths = predicted_edges["score_norm"].to_numpy(dtype=float)
    color_strengths = strengths
    colors = [
        MI_CMAPS[label](color_strength)
        for label, color_strength in zip(
            predicted_edges["plot_label"], color_strengths
        )
    ]
    linewidths = EDGE_LINEWIDTH * (0.65 + 0.85 * strengths)
    ax.scatter(
        spatial_location[:, 0], spatial_location[:, 1],
        c="lightgrey", s=CELL_SIZE, zorder=0,
    )
    _draw_arrows(
        ax,
        predicted_edges,
        colors=colors,
        linewidths=linewidths,
    )
    predicted_pairs = pd.MultiIndex.from_frame(
        predicted_edges[["sender_index", "receiver_index"]]
    )
    truth_pairs = pd.MultiIndex.from_frame(
        truth_edge_df[["sender_index", "receiver_index"]]
    )
    off_ring_count = int((~predicted_pairs.isin(truth_pairs)).sum())
    print(
        f"[{method_name}] candidate tissue edges={len(edge_scores_df)}, "
        f"displayed top-k edges={len(predicted_edges)}, "
        f"displayed outside the ground-truth ring={off_ring_count}"
    )
    ax.set_title(method_name)
    format_spatial_axis(ax)


def build_tp_fp_edge_table(edge_scores_df):
    """Classify every tissue edge using its higher inferred MI activity."""
    score_matrix = np.nan_to_num(
        edge_scores_df[MI_NAMES].to_numpy(dtype=float),
        nan=0.0, posinf=1.0, neginf=0.0,
    )
    predicted_mi = np.asarray(MI_NAMES)[np.argmax(score_matrix, axis=1)]
    interacting_activity = np.clip(np.max(score_matrix, axis=1), 0.0, 1.0)
    color_strength = interacting_activity ** 2
    truth_is_interaction = np.isin(truth_labels, MI_NAMES)
    is_true_positive = truth_is_interaction & (predicted_mi == truth_labels)

    edge_table = reference_pairs.copy()
    edge_table["predicted_mi"] = predicted_mi
    edge_table["interacting_activity"] = interacting_activity
    edge_table["color_strength"] = color_strength
    edge_table["classification"] = np.where(
        is_true_positive, "True positive", "False positive"
    )
    edge_table["signed_activity"] = np.where(
        is_true_positive, color_strength, -color_strength
    )
    return edge_table.sort_values("signed_activity").reset_index(drop=True)


def plot_method_tp_fp(ax, method_name, edge_scores_df):
    tp_fp_edges = build_tp_fp_edge_table(edge_scores_df)
    signed_activity = tp_fp_edges["signed_activity"].to_numpy(dtype=float)
    colors = TP_FP_CMAP(TP_FP_NORM(signed_activity))
    linewidths = np.full(len(tp_fp_edges), EDGE_LINEWIDTH)
    ax.scatter(
        spatial_location[:, 0], spatial_location[:, 1],
        c="lightgrey", s=CELL_SIZE, zorder=0,
    )
    _draw_arrows(
        ax, tp_fp_edges, colors=colors, linewidths=linewidths
    )
    n_true_positive = int((tp_fp_edges["classification"] == "True positive").sum())
    n_false_positive = int(len(tp_fp_edges) - n_true_positive)
    print(
        f"[{method_name}] true-positive edges={n_true_positive}, "
        f"false-positive edges={n_false_positive}"
    )
    ax.set_title(method_name)
    format_spatial_axis(ax)


In [ ]:
# ------------------------------------------------------------
# 4. Inferred MI activity by ground-truth edge type
# ------------------------------------------------------------
# This analysis uses SpiderNet scores for the single setting/experiment
# selected at the top of this notebook.
if "SpiderNet" not in method_edge_scores:
    raise ValueError("SpiderNet edge scores are required for the inferred-activity panels.")

EDGE_TYPE_ORDER = ["MI-1", "MI-2", "Non-int."]
EDGE_TYPE_RENAME = {"non-interaction": "Non-int."}
EDGE_EDGE_COLORS = {
    "MI-1": "#ED2187",
    "MI-2": "#FAA51B",
    "Non-int.": "#8A8A8A",
}
EDGE_FILL_COLORS = {
    "MI-1": "#E79AB2",
    "MI-2": "#FAD08A",
    "Non-int.": "#C9CED3",
}

spidernet_scores = method_edge_scores["SpiderNet"].copy().reset_index(drop=True)
if len(spidernet_scores) != len(edgemeta_data):
    raise ValueError(
        f"SpiderNet score/truth length mismatch: {len(spidernet_scores)} scores vs "
        f"{len(edgemeta_data)} ground-truth edges."
    )

edge_activity_df = spidernet_scores[["sender_index", "receiver_index", "MI-1", "MI-2"]].copy()
edge_activity_df["Ground truth edge type"] = (
    edgemeta_data["MetaItype"].astype(str).replace(EDGE_TYPE_RENAME).to_numpy()
)
edge_activity_df = edge_activity_df[
    edge_activity_df["Ground truth edge type"].isin(EDGE_TYPE_ORDER)
].copy()
edge_activity_df["Ground truth edge type"] = pd.Categorical(
    edge_activity_df["Ground truth edge type"],
    categories=EDGE_TYPE_ORDER,
    ordered=True,
)

panel_specs = [
    ("MI-1", "Inferred MI-1 activity", "MI-1", ["MI-2", "Non-int."]),
    ("MI-2", "Inferred MI-2 activity", "MI-2", ["MI-1", "Non-int."]),
]
median_rows = []
test_rows = []

for score_name, y_label, reference_group, comparison_groups in panel_specs:
    for group_name in EDGE_TYPE_ORDER:
        values = edge_activity_df.loc[
            edge_activity_df["Ground truth edge type"] == group_name, score_name
        ].dropna().to_numpy(dtype=float)
        median_rows.append({
            "panel": score_name,
            "group": group_name,
            "median": float(np.median(values)) if values.size else np.nan,
            "n": int(values.size),
        })

    reference_values = edge_activity_df.loc[
        edge_activity_df["Ground truth edge type"] == reference_group, score_name
    ].dropna().to_numpy(dtype=float)
    for comparison_group in comparison_groups:
        comparison_values = edge_activity_df.loc[
            edge_activity_df["Ground truth edge type"] == comparison_group, score_name
        ].dropna().to_numpy(dtype=float)
        if reference_values.size == 0 or comparison_values.size == 0:
            statistic, p_value = np.nan, np.nan
        else:
            statistic, p_value = ranksums(
                reference_values, comparison_values, alternative="two-sided"
            )
        test_rows.append({
            "panel": score_name,
            "group_1": reference_group,
            "group_2": comparison_group,
            "alternative": "two-sided",
            "statistic": statistic,
            "p_value": p_value,
            "n_1": int(reference_values.size),
            "n_2": int(comparison_values.size),
        })

edge_activity_medians = pd.DataFrame(median_rows)
edge_activity_tests = pd.DataFrame(test_rows)

print("All group medians:")
print(edge_activity_medians.to_string(index=False))
print("\nTwo-sided Wilcoxon rank-sum tests:")
print(edge_activity_tests.to_string(index=False))

def _format_pvalue(p_value):
    if not np.isfinite(p_value):
        return "p = NA"
    return f"p = {p_value:.2e}" if p_value < 0.001 else f"p = {p_value:.3f}"


def _add_pvalue_bracket(ax, x1, x2, y, height, label):
    ax.plot(
        [x1, x1, x2, x2], [y, y + height, y + height, y],
        color="black", linewidth=0.7, clip_on=False,
    )
    ax.text(
        (x1 + x2) / 2, y + height, label,
        ha="center", va="bottom", fontsize=6.5, clip_on=False,
    )


fig_activity, axes_activity = plt.subplots(
    1, 2, figsize=(4.3, 2.45), constrained_layout=True
)
for ax, (score_name, y_label, reference_group, comparison_groups) in zip(
    axes_activity, panel_specs
):
    values_by_group = [
        edge_activity_df.loc[
            edge_activity_df["Ground truth edge type"] == group_name, score_name
        ].dropna().to_numpy(dtype=float)
        for group_name in EDGE_TYPE_ORDER
    ]
    if any(values.size == 0 for values in values_by_group):
        empty_groups = [
            group_name for group_name, values in zip(EDGE_TYPE_ORDER, values_by_group)
            if values.size == 0
        ]
        raise ValueError(f"No {score_name} values for groups: {empty_groups}")

    positions = np.arange(len(EDGE_TYPE_ORDER))
    violin_parts = ax.violinplot(
        values_by_group, positions=positions, widths=0.78,
        showmeans=False, showmedians=False, showextrema=False,
    )
    for body, group_name in zip(violin_parts["bodies"], EDGE_TYPE_ORDER):
        body.set_facecolor(EDGE_FILL_COLORS[group_name])
        body.set_edgecolor(EDGE_EDGE_COLORS[group_name])
        body.set_linewidth(0.8)
        body.set_alpha(1.0)

    ax.boxplot(
        values_by_group, positions=positions, widths=0.22, patch_artist=True,
        showfliers=False, showcaps=False, whis=1.5,
        medianprops=dict(color="white", linewidth=1.0),
        boxprops=dict(facecolor="#58595B", edgecolor="#58595B", linewidth=0.7),
        whiskerprops=dict(color="#58595B", linewidth=0.7),
    )

    panel_tests = edge_activity_tests[edge_activity_tests["panel"] == score_name]
    data_min = min(float(np.nanmin(values)) for values in values_by_group)
    data_max = max(float(np.nanmax(values)) for values in values_by_group)
    data_range = max(data_max - data_min, 0.1)
    bracket_height = 0.035 * data_range
    bracket_gap = 0.13 * data_range
    for comparison_idx, comparison_group in enumerate(comparison_groups):
        test_row = panel_tests[panel_tests["group_2"] == comparison_group].iloc[0]
        x1 = EDGE_TYPE_ORDER.index(reference_group)
        x2 = EDGE_TYPE_ORDER.index(comparison_group)
        bracket_y = data_max + (comparison_idx + 0.35) * bracket_gap
        _add_pvalue_bracket(
            ax, x1, x2, bracket_y, bracket_height,
            _format_pvalue(test_row["p_value"]),
        )

    ax.set_ylim(data_min - 0.04 * data_range, data_max + 2.25 * bracket_gap)
    ax.set_xticks(positions)
    ax.set_xticklabels(EDGE_TYPE_ORDER)
    ax.set_xlabel("Ground truth edge type")
    ax.set_ylabel(y_label)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", direction="out")
    ax.grid(False)

if SAVE_FIGURE:
    out_dir = SAMPLE_OUT_DIR
    out_dir.mkdir(parents=True, exist_ok=True)
    output_stem = f"Inferred_MI_activity_by_ground_truth_{setting_name}_Experiment_{EXPERIMENT_INDEX}"
    fig_activity.savefig(out_dir / f"{output_stem}.pdf", bbox_inches="tight")
    fig_activity.savefig(out_dir / f"{output_stem}.png", dpi=300, bbox_inches="tight")
    edge_activity_df.to_csv(out_dir / f"{output_stem}_edge_scores.csv", index=False)
    edge_activity_medians.to_csv(out_dir / f"{output_stem}_medians.csv", index=False)
    edge_activity_tests.to_csv(out_dir / f"{output_stem}_wilcoxon_rank_sum.csv", index=False)
    print("Saved inferred-activity figure and statistics to:", out_dir)

plt.show()


In [ ]:
# ------------------------------------------------------------
# 5. Spatial map of top-ranked MI predictions
# ------------------------------------------------------------
selected_methods = [
    method_name for method_name in METHODS_TO_SHOW
    if method_name in method_edge_scores
]
if not selected_methods:
    raise ValueError("No method outputs were available for plotting.")

truth_edge_df = build_truth_edge_table()
panel_names = selected_methods.copy()
if SHOW_GROUND_TRUTH:
    panel_names.insert(0, "Ground truth")

n_panels = len(panel_names)
n_cols = min(3, n_panels)
n_rows = int(np.ceil(n_panels / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.4 * n_cols, 4.1 * n_rows))
axes = np.atleast_1d(axes).reshape(n_rows, n_cols)

for ax in axes.flat:
    ax.axis("off")

for ax, panel_name in zip(axes.flat, panel_names):
    ax.axis("on")
    if panel_name == "Ground truth":
        plot_ground_truth(ax, truth_edge_df)
    else:
        plot_method(
            ax, panel_name, method_edge_scores[panel_name], truth_edge_df
        )

truth_handles = [
    Line2D([0], [0], color=MI_COLORS[mi_name], lw=2, label=mi_name)
    for mi_name in MI_NAMES
]
fig.legend(
    handles=truth_handles,
    title="Predicted MI",
    loc="lower center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 0.005),
)
fig.suptitle(
    f"In-situ recovery of simulated MI structure | "
    f"{setting_name} | Experiment_{EXPERIMENT_INDEX}",
    y=0.995,
)
fig.subplots_adjust(top=0.92, bottom=0.10, wspace=0.12, hspace=0.16)

if SAVE_FIGURE:
    out_dir = SAMPLE_OUT_DIR
    out_dir.mkdir(parents=True, exist_ok=True)
    spatial_output_path = (
        out_dir
        / f"Insitu_MIall_compare_five_methods_{setting_name}_Experiment_{EXPERIMENT_INDEX}.pdf"
    )
    fig.savefig(spatial_output_path)
    print("Saved spatial comparison to:", spatial_output_path)

plt.show()


# ------------------------------------------------------------
# 6. Spatial distribution of true- and false-positive edges
# ------------------------------------------------------------
n_tp_fp_panels = len(selected_methods)
n_tp_fp_cols = min(2, n_tp_fp_panels)
n_tp_fp_rows = int(np.ceil(n_tp_fp_panels / n_tp_fp_cols))
fig_tp_fp, axes_tp_fp = plt.subplots(
    n_tp_fp_rows, n_tp_fp_cols,
    figsize=(4.8 * n_tp_fp_cols, 4.25 * n_tp_fp_rows),
)
axes_tp_fp = np.atleast_1d(axes_tp_fp).reshape(
    n_tp_fp_rows, n_tp_fp_cols
)

for ax in axes_tp_fp.flat:
    ax.axis("off")

active_tp_fp_axes = []
for ax, method_name in zip(axes_tp_fp.flat, selected_methods):
    ax.axis("on")
    plot_method_tp_fp(ax, method_name, method_edge_scores[method_name])
    active_tp_fp_axes.append(ax)

tp_fp_colorbar = fig_tp_fp.colorbar(
    ScalarMappable(norm=TP_FP_NORM, cmap=TP_FP_CMAP),
    ax=active_tp_fp_axes,
    orientation="horizontal",
    fraction=0.035,
    pad=0.07,
    aspect=35,
)
tp_fp_colorbar.set_ticks([-1, 0, 1])
tp_fp_colorbar.set_ticklabels([
    "False positive: high activity",
    "Low interacting activity",
    "True positive: high activity",
])
tp_fp_colorbar.set_label("Cubed maximum inferred interacting activity")
fig_tp_fp.suptitle(
    f"Spatial distribution of true- and false-positive interactions | "
    f"{setting_name} | Experiment_{EXPERIMENT_INDEX}",
    y=0.995,
)
fig_tp_fp.subplots_adjust(
    top=0.93, bottom=0.12, wspace=0.10, hspace=0.14
)

if SAVE_FIGURE:
    tp_fp_output_path = (
        out_dir
        / f"Insitu_MIall_true_false_positive_{setting_name}_Experiment_{EXPERIMENT_INDEX}.pdf"
    )
    fig_tp_fp.savefig(tp_fp_output_path)
    print("Saved true/false-positive spatial comparison to:", tp_fp_output_path)

plt.show()
